In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('crash_data_new.csv')
df.head()

,CRASH_RECORD_ID,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,ALIGNMENT,...,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,IDOT_CONTROL_NO,LATITUDE,LONGITUDE,LOCATION
0,419e0f4348b4274d902d84bd7aa58f5fa9c10820b77abc...,05/19/2026 11:15:00 PM,45,STOP SIGN/FLASHER,FUNCTIONING IMPROPERLY,CLEAR,DARKNESS,TURNING,DIVIDED - W/MEDIAN (NOT RAISED),STRAIGHT AND LEVEL,...,0.0,0.0,2.0,0.0,23,3,X004237573,41.781851,-87.575475,POINT (-87.575474638028 41.781851295939)
1,1471094d2f2a497653584d887ed73cddcdfaa35fd02b20...,05/20/2026 07:51:00 AM,30,STOP SIGN/FLASHER,FUNCTIONING PROPERLY,CLEAR,DAYLIGHT,PEDESTRIAN,FOUR WAY,STRAIGHT AND LEVEL,...,2.0,0.0,2.0,0.0,7,4,X004237669,41.894349,-87.662208,POINT (-87.662207823341 41.89434904862)
2,1c99690e19bd557cc6ad0674668dc492ab979c4e63a279...,05/20/2026 02:00:00 AM,30,UNKNOWN,UNKNOWN,CLEAR,"DARKNESS, LIGHTED ROAD",FIXED OBJECT,NOT DIVIDED,STRAIGHT AND LEVEL,...,0.0,0.0,1.0,0.0,2,4,X004237255,41.759085,-87.567448,POINT (-87.567447849458 41.759085475865)
3,d4617f749ee48ca7b2b74cd96171a7ab3d786cf2fcbda5...,05/18/2026 07:45:00 AM,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,PARKED MOTOR VEHICLE,NOT DIVIDED,STRAIGHT AND LEVEL,...,0.0,0.0,1.0,0.0,7,2,X004237264,41.654285,-87.603500,POINT (-87.603500025118 41.654285003672)
4,7b69b249bb80528d33287ece0d4e54eb5d26c8d7b27ee5...,05/19/2026 12:20:00 PM,25,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,FIXED OBJECT,ONE-WAY,STRAIGHT AND LEVEL,...,0.0,0.0,1.0,0.0,12,3,X004237216,41.773470,-87.712459,POINT (-87.712459184495 41.773470407719)


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1019818 entries, 0 to 1019817
Data columns (total 38 columns):
 #   Column                         Non-Null Count    Dtype  
---  ------                         --------------    -----  
 0   CRASH_RECORD_ID                1019818 non-null  str    
 1   CRASH_DATE                     1019818 non-null  str    
 2   POSTED_SPEED_LIMIT             1019818 non-null  int64  
 3   TRAFFIC_CONTROL_DEVICE         1019818 non-null  str    
 4   DEVICE_CONDITION               1019818 non-null  str    
 5   WEATHER_CONDITION              1019818 non-null  str    
 6   LIGHTING_CONDITION             1019818 non-null  str    
 7   FIRST_CRASH_TYPE               1019818 non-null  str    
 8   TRAFFICWAY_TYPE                1019818 non-null  str    
 9   ALIGNMENT                      1019818 non-null  str    
 10  ROADWAY_SURFACE_COND           1019818 non-null  str    
 11  ROAD_DEFECT                    1019818 non-null  str    
 12  REPORT_TYPE              

In [4]:
import pandas as pd
import numpy as np

# =========================
# 1. Carregar base
# =========================

# Padronizar nomes das colunas
df.columns = df.columns.str.upper()

# =========================
# 2. Manter apenas colunas disponíveis no seu trabalho
# =========================

cols_crash = [
    'CRASH_RECORD_ID',
    'CRASH_DATE',
    'POSTED_SPEED_LIMIT',
    'TRAFFIC_CONTROL_DEVICE',
    'DEVICE_CONDITION',
    'WEATHER_CONDITION',
    'LIGHTING_CONDITION',
    'FIRST_CRASH_TYPE',
    'TRAFFICWAY_TYPE',
    'ALIGNMENT',
    'ROADWAY_SURFACE_COND',
    'ROAD_DEFECT',
    'REPORT_TYPE',
    'CRASH_TYPE',
    'DAMAGE',
    'DATE_POLICE_NOTIFIED',
    'PRIM_CONTRIBUTORY_CAUSE',
    'SEC_CONTRIBUTORY_CAUSE',
    'STREET_NO',
    'STREET_DIRECTION',
    'STREET_NAME',
    'BEAT_OF_OCCURRENCE',
    'NUM_UNITS',
    'CRASH_MONTH',
    'MOST_SEVERE_INJURY',
    'INJURIES_TOTAL',
    'INJURIES_FATAL',
    'INJURIES_INCAPACITATING',
    'INJURIES_NON_INCAPACITATING',
    'INJURIES_REPORTED_NOT_EVIDENT',
    'INJURIES_NO_INDICATION',
    'INJURIES_UNKNOWN',
    'CRASH_HOUR',
    'CRASH_DAY_OF_WEEK',
    'IDOT_CONTROL_NO',
    'LATITUDE',
    'LONGITUDE',
    'LOCATION'
]

df = df[[col for col in cols_crash if col in df.columns]]

# =========================
# 3. Converter datas
# =========================

df['CRASH_DATE'] = pd.to_datetime(df['CRASH_DATE'], errors='coerce')

if 'DATE_POLICE_NOTIFIED' in df.columns:
    df['DATE_POLICE_NOTIFIED'] = pd.to_datetime(df['DATE_POLICE_NOTIFIED'], errors='coerce')

# Remover registros sem data
df = df.dropna(subset=['CRASH_DATE'])

# =========================
# 4. Criar variáveis temporais
# =========================

df['ANO'] = df['CRASH_DATE'].dt.year
df['MES'] = df['CRASH_DATE'].dt.month
df['ANO_MES'] = df['CRASH_DATE'].dt.to_period('M').astype(str)

# Garantir hora e dia da semana
df['CRASH_HOUR'] = df['CRASH_HOUR'].fillna(df['CRASH_DATE'].dt.hour)
df['CRASH_DAY_OF_WEEK'] = df['CRASH_DAY_OF_WEEK'].fillna(df['CRASH_DATE'].dt.dayofweek + 1)

# =========================
# 5. Filtrar período
# =========================
# A documentação informa que dados citywide só ficam completos a partir de setembro de 2017.
# Para evitar ano incompleto, começar em 2018.

df = df[df['ANO'] >= 2018]

# =========================
# 6. Remover duplicatas
# =========================

df = df.drop_duplicates(subset=['CRASH_RECORD_ID'])

# =========================
# 7. Limpar localização
# =========================

df = df.dropna(subset=['LATITUDE', 'LONGITUDE'])

df = df[
    (df['LATITUDE'].between(41.6, 42.1)) &
    (df['LONGITUDE'].between(-88.0, -87.4))
]

# =========================
# 8. Limpar BEAT_OF_OCCURRENCE
# =========================

df = df.dropna(subset=['BEAT_OF_OCCURRENCE'])

df['BEAT_OF_OCCURRENCE'] = (
    df['BEAT_OF_OCCURRENCE']
    .astype(float)
    .astype(int)
    .astype(str)
)

# =========================
# 9. Criar nome da rua
# =========================

df['STREET_DIRECTION'] = df['STREET_DIRECTION'].fillna('')
df['STREET_NAME'] = df['STREET_NAME'].fillna('UNKNOWN')

df['STREET_FULL'] = (
    df['STREET_DIRECTION'].astype(str).str.strip() + ' ' +
    df['STREET_NAME'].astype(str).str.strip()
).str.strip()

# =========================
# 10. Tratar numéricas de lesão
# =========================

injury_cols = [
    'INJURIES_TOTAL',
    'INJURIES_FATAL',
    'INJURIES_INCAPACITATING',
    'INJURIES_NON_INCAPACITATING',
    'INJURIES_REPORTED_NOT_EVIDENT',
    'INJURIES_NO_INDICATION',
    'INJURIES_UNKNOWN'
]

for col in injury_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# =========================
# 11. Tratar numéricas gerais
# =========================

numeric_cols = [
    'POSTED_SPEED_LIMIT',
    'NUM_UNITS',
    'CRASH_MONTH',
    'CRASH_HOUR',
    'CRASH_DAY_OF_WEEK',
    'LATITUDE',
    'LONGITUDE'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['POSTED_SPEED_LIMIT'] = df['POSTED_SPEED_LIMIT'].fillna(df['POSTED_SPEED_LIMIT'].median())
df['NUM_UNITS'] = df['NUM_UNITS'].fillna(df['NUM_UNITS'].median())

# =========================
# 12. Tratar categóricas
# =========================

categorical_cols = [
    'TRAFFIC_CONTROL_DEVICE',
    'DEVICE_CONDITION',
    'WEATHER_CONDITION',
    'LIGHTING_CONDITION',
    'FIRST_CRASH_TYPE',
    'TRAFFICWAY_TYPE',
    'ALIGNMENT',
    'ROADWAY_SURFACE_COND',
    'ROAD_DEFECT',
    'REPORT_TYPE',
    'CRASH_TYPE',
    'DAMAGE',
    'PRIM_CONTRIBUTORY_CAUSE',
    'SEC_CONTRIBUTORY_CAUSE',
    'MOST_SEVERE_INJURY'
]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna('UNKNOWN').astype(str).str.upper().str.strip()

# =========================
# 13. Criar indicadores auxiliares
# =========================

df['TEVE_LESAO'] = df['INJURIES_TOTAL'] > 0
df['TEVE_MORTE'] = df['INJURIES_FATAL'] > 0
df['TEVE_LESAO_GRAVE'] = (
    (df['INJURIES_FATAL'] > 0) |
    (df['INJURIES_INCAPACITATING'] > 0)
)

def classificar_periodo(hora):
    if 0 <= hora <= 5:
        return 'MADRUGADA'
    elif 6 <= hora <= 11:
        return 'MANHA'
    elif 12 <= hora <= 17:
        return 'TARDE'
    else:
        return 'NOITE'

df['PERIODO_DIA'] = df['CRASH_HOUR'].apply(classificar_periodo)

# Indicadores contextuais
df['ILUMINACAO_RISCO'] = df['LIGHTING_CONDITION'].isin([
    'DARKNESS',
    'DARKNESS, LIGHTED ROAD',
    'DUSK',
    'DAWN'
])

df['PISTA_RISCO'] = ~df['ROADWAY_SURFACE_COND'].isin([
    'DRY',
    'UNKNOWN'
])

df['DEFEITO_VIA'] = ~df['ROAD_DEFECT'].isin([
    'NO DEFECTS',
    'UNKNOWN'
])

df['DISPOSITIVO_PROBLEMA'] = ~df['DEVICE_CONDITION'].isin([
    'FUNCTIONING PROPERLY',
    'NO CONTROLS',
    'UNKNOWN'
])

df['CLIMA_RISCO'] = ~df['WEATHER_CONDITION'].isin([
    'CLEAR',
    'UNKNOWN'
])

# =========================
# 14. Salvar base limpa
# =========================

df.to_csv("crashes_limpa.csv", index=False)

print(df.shape)
print(df.head())

/tmp/ipykernel_99754/1554959263.py:62: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['CRASH_DATE'] = pd.to_datetime(df['CRASH_DATE'], errors='coerce')


(884712, 51)
                                     CRASH_RECORD_ID          CRASH_DATE  \
0  419e0f4348b4274d902d84bd7aa58f5fa9c10820b77abc... 2026-05-19 23:15:00   
1  1471094d2f2a497653584d887ed73cddcdfaa35fd02b20... 2026-05-20 07:51:00   
2  1c99690e19bd557cc6ad0674668dc492ab979c4e63a279... 2026-05-20 02:00:00   
3  d4617f749ee48ca7b2b74cd96171a7ab3d786cf2fcbda5... 2026-05-18 07:45:00   
4  7b69b249bb80528d33287ece0d4e54eb5d26c8d7b27ee5... 2026-05-19 12:20:00   

   POSTED_SPEED_LIMIT TRAFFIC_CONTROL_DEVICE        DEVICE_CONDITION  \
0                  45      STOP SIGN/FLASHER  FUNCTIONING IMPROPERLY   
1                  30      STOP SIGN/FLASHER    FUNCTIONING PROPERLY   
2                  30                UNKNOWN                 UNKNOWN   
3                  30            NO CONTROLS             NO CONTROLS   
4                  25            NO CONTROLS             NO CONTROLS   

  WEATHER_CONDITION      LIGHTING_CONDITION      FIRST_CRASH_TYPE  \
0             CLEAR         

In [12]:
df["CRASH_DATE"] = pd.to_datetime(df["CRASH_DATE"], errors="coerce")
df = df.dropna(subset=["CRASH_DATE"])

# Criar mês de referência
df["MES_REF"] = df["CRASH_DATE"].dt.to_period("M").dt.to_timestamp()

# Garantir que o beat seja string
df["BEAT_OF_OCCURRENCE"] = df["BEAT_OF_OCCURRENCE"].astype(str)

# ============================================================
# 2. Garantir que variáveis booleanas estejam corretas
# ============================================================

bool_cols = [
    "TEVE_LESAO",
    "TEVE_MORTE",
    "TEVE_LESAO_GRAVE",
    "ILUMINACAO_RISCO",
    "PISTA_RISCO",
    "DEFEITO_VIA",
    "DISPOSITIVO_PROBLEMA",
    "CLIMA_RISCO"
]

for col in bool_cols:
    if col in df.columns:
        if df[col].dtype == "object":
            df[col] = (
                df[col]
                .astype(str)
                .str.upper()
                .map({"TRUE": 1, "FALSE": 0})
                .fillna(0)
                .astype(int)
            )
        else:
            df[col] = df[col].fillna(False).astype(int)

# Criar indicadores auxiliares
df["ACIDENTE_NOITE"] = df["PERIODO_DIA"].isin(["NOITE", "MADRUGADA"]).astype(int)
df["ACIDENTE_FIM_SEMANA"] = (df["CRASH_DATE"].dt.dayofweek >= 5).astype(int)

# Garantir numéricas
num_cols = [
    "POSTED_SPEED_LIMIT",
    "NUM_UNITS",
    "INJURIES_TOTAL",
    "INJURIES_FATAL",
    "INJURIES_INCAPACITATING"
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# ============================================================
# 3. Agregar por BEAT + mês
# ============================================================

beat_mes = (
    df.groupby(["BEAT_OF_OCCURRENCE", "MES_REF"])
    .agg(
        total_acidentes_mes=("CRASH_RECORD_ID", "count"),

        acidentes_noite_mes=("ACIDENTE_NOITE", "sum"),
        acidentes_fim_semana_mes=("ACIDENTE_FIM_SEMANA", "sum"),

        iluminacao_risco_mes=("ILUMINACAO_RISCO", "sum"),
        pista_risco_mes=("PISTA_RISCO", "sum"),
        defeito_via_mes=("DEFEITO_VIA", "sum"),
        dispositivo_problema_mes=("DISPOSITIVO_PROBLEMA", "sum"),
        clima_risco_mes=("CLIMA_RISCO", "sum"),

        velocidade_soma_mes=("POSTED_SPEED_LIMIT", "sum"),
        num_units_soma_mes=("NUM_UNITS", "sum"),

        acidentes_com_lesao_mes=("TEVE_LESAO", "sum"),
        acidentes_graves_mes=("TEVE_LESAO_GRAVE", "sum"),
        acidentes_fatais_mes=("TEVE_MORTE", "sum"),

        total_lesoes_mes=("INJURIES_TOTAL", "sum"),
        total_lesoes_graves_mes=("INJURIES_INCAPACITATING", "sum"),
        total_mortes_mes=("INJURIES_FATAL", "sum"),

        lat_media_mes=("LATITUDE", "mean"),
        lon_media_mes=("LONGITUDE", "mean")
    )
    .reset_index()
)

# ============================================================
# 4. Completar meses sem acidentes
# ============================================================
# Isso é importante.
# Se um beat não teve acidente em determinado mês, ele precisa aparecer com zero.
# Caso contrário, o modelo só aprenderia meses em que houve acidente.

todos_beats = beat_mes["BEAT_OF_OCCURRENCE"].unique()
todos_meses = pd.date_range(
    beat_mes["MES_REF"].min(),
    beat_mes["MES_REF"].max(),
    freq="MS"
)

indice_completo = pd.MultiIndex.from_product(
    [todos_beats, todos_meses],
    names=["BEAT_OF_OCCURRENCE", "MES_REF"]
)

beat_mes = (
    beat_mes
    .set_index(["BEAT_OF_OCCURRENCE", "MES_REF"])
    .reindex(indice_completo)
    .reset_index()
)

# Colunas mensais que devem virar zero quando não houve acidente
cols_zero = [
    "total_acidentes_mes",
    "acidentes_noite_mes",
    "acidentes_fim_semana_mes",
    "iluminacao_risco_mes",
    "pista_risco_mes",
    "defeito_via_mes",
    "dispositivo_problema_mes",
    "clima_risco_mes",
    "velocidade_soma_mes",
    "num_units_soma_mes",
    "acidentes_com_lesao_mes",
    "acidentes_graves_mes",
    "acidentes_fatais_mes",
    "total_lesoes_mes",
    "total_lesoes_graves_mes",
    "total_mortes_mes"
]

beat_mes[cols_zero] = beat_mes[cols_zero].fillna(0)

# Coordenadas médias por beat, para usar no mapa caso você não use GeoJSON
coords_beat = (
    df.groupby("BEAT_OF_OCCURRENCE")
    .agg(
        lat_media_beat=("LATITUDE", "mean"),
        lon_media_beat=("LONGITUDE", "mean")
    )
    .reset_index()
)

beat_mes = beat_mes.merge(coords_beat, on="BEAT_OF_OCCURRENCE", how="left")

beat_mes["lat_media_mes"] = beat_mes["lat_media_mes"].fillna(beat_mes["lat_media_beat"])
beat_mes["lon_media_mes"] = beat_mes["lon_media_mes"].fillna(beat_mes["lon_media_beat"])

# Salvar base mensal simples
beat_mes.to_csv("base_beat_mes.csv", index=False)

print("Base beat-mês criada:")
print(beat_mes.shape)
print(beat_mes.head())

Base beat-mês criada:
(27744, 22)
  BEAT_OF_OCCURRENCE    MES_REF  total_acidentes_mes  acidentes_noite_mes  \
0               1011 2018-01-01                 22.0                  8.0   
1               1011 2018-02-01                 21.0                  6.0   
2               1011 2018-03-01                 18.0                  6.0   
3               1011 2018-04-01                 18.0                 11.0   
4               1011 2018-05-01                 32.0                 12.0   

   acidentes_fim_semana_mes  iluminacao_risco_mes  pista_risco_mes  \
0                       3.0                  10.0             11.0   
1                       4.0                   8.0              8.0   
2                       6.0                   7.0              1.0   
3                       5.0                  11.0              2.0   
4                       8.0                  11.0              4.0   

   defeito_via_mes  dispositivo_problema_mes  clima_risco_mes  ...  \
0           

In [14]:
beat_mes.info()

<class 'pandas.DataFrame'>
RangeIndex: 27744 entries, 0 to 27743
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   BEAT_OF_OCCURRENCE        27744 non-null  str           
 1   MES_REF                   27744 non-null  datetime64[us]
 2   total_acidentes_mes       27744 non-null  float64       
 3   acidentes_noite_mes       27744 non-null  float64       
 4   acidentes_fim_semana_mes  27744 non-null  float64       
 5   iluminacao_risco_mes      27744 non-null  float64       
 6   pista_risco_mes           27744 non-null  float64       
 7   defeito_via_mes           27744 non-null  float64       
 8   dispositivo_problema_mes  27744 non-null  float64       
 9   clima_risco_mes           27744 non-null  float64       
 10  velocidade_soma_mes       27744 non-null  float64       
 11  num_units_soma_mes        27744 non-null  float64       
 12  acidentes_com_lesao_mes   277

In [15]:
beat_mes.head()

,BEAT_OF_OCCURRENCE,MES_REF,total_acidentes_mes,acidentes_noite_mes,acidentes_fim_semana_mes,iluminacao_risco_mes,pista_risco_mes,defeito_via_mes,dispositivo_problema_mes,clima_risco_mes,...,acidentes_com_lesao_mes,acidentes_graves_mes,acidentes_fatais_mes,total_lesoes_mes,total_lesoes_graves_mes,total_mortes_mes,lat_media_mes,lon_media_mes,lat_media_beat,lon_media_beat
0,1011,2018-01-01,22.0,8.0,3.0,10.0,11.0,0.0,1.0,7.0,...,6.0,0.0,0.0,6.0,0.0,0.0,41.863518,-87.723500,41.8643,-87.725041
1,1011,2018-02-01,21.0,6.0,4.0,8.0,8.0,0.0,0.0,9.0,...,2.0,0.0,0.0,3.0,0.0,0.0,41.864455,-87.724508,41.8643,-87.725041
2,1011,2018-03-01,18.0,6.0,6.0,7.0,1.0,0.0,2.0,1.0,...,5.0,1.0,0.0,6.0,1.0,0.0,41.864083,-87.727347,41.8643,-87.725041
3,1011,2018-04-01,18.0,11.0,5.0,11.0,2.0,1.0,0.0,3.0,...,3.0,2.0,0.0,3.0,2.0,0.0,41.864173,-87.722843,41.8643,-87.725041
4,1011,2018-05-01,32.0,12.0,8.0,11.0,4.0,1.0,1.0,3.0,...,8.0,3.0,0.0,8.0,3.0,0.0,41.864514,-87.725852,41.8643,-87.725041


In [16]:


base = beat_mes.copy()

base["MES_REF"] = pd.to_datetime(base["MES_REF"], errors="coerce")
base = base.sort_values(["BEAT_OF_OCCURRENCE", "MES_REF"])

# ============================================================
# 2. Criar score futuro mensal
# ============================================================
# Esse score usa lesões e mortes, mas SOMENTE para criar o alvo futuro.
# Ele não será usado como feature de entrada do modelo.

base["score_prioridade_mes"] = (
    base["total_acidentes_mes"]
    + 3 * base["acidentes_com_lesao_mes"]
    + 8 * base["total_lesoes_graves_mes"]
    + 15 * base["total_mortes_mes"]
)

# ============================================================
# 3. Criar features dos últimos 6 meses
# ============================================================

cols_soma_6m = [
    "total_acidentes_mes",
    "acidentes_noite_mes",
    "acidentes_fim_semana_mes",
    "iluminacao_risco_mes",
    "pista_risco_mes",
    "defeito_via_mes",
    "dispositivo_problema_mes",
    "clima_risco_mes",
    "velocidade_soma_mes",
    "num_units_soma_mes"
]

for col in cols_soma_6m:
    base[col.replace("_mes", "_ult_6m")] = (
        base
        .groupby("BEAT_OF_OCCURRENCE")[col]
        .transform(lambda s: s.shift(1).rolling(window=6, min_periods=3).sum())
    )

# Média de acidentes mensal nos últimos 6 meses
base["media_acidentes_mensal_ult_6m"] = base["total_acidentes_ult_6m"] / 6

# Percentuais dos últimos 6 meses
def safe_div(num, den):
    return np.where(den > 0, num / den, 0)

base["perc_acidentes_noite_ult_6m"] = safe_div(
    base["acidentes_noite_ult_6m"],
    base["total_acidentes_ult_6m"]
)

base["perc_acidentes_fim_semana_ult_6m"] = safe_div(
    base["acidentes_fim_semana_ult_6m"],
    base["total_acidentes_ult_6m"]
)

base["perc_iluminacao_risco_ult_6m"] = safe_div(
    base["iluminacao_risco_ult_6m"],
    base["total_acidentes_ult_6m"]
)

base["perc_pista_risco_ult_6m"] = safe_div(
    base["pista_risco_ult_6m"],
    base["total_acidentes_ult_6m"]
)

base["perc_defeito_via_ult_6m"] = safe_div(
    base["defeito_via_ult_6m"],
    base["total_acidentes_ult_6m"]
)

base["perc_dispositivo_problema_ult_6m"] = safe_div(
    base["dispositivo_problema_ult_6m"],
    base["total_acidentes_ult_6m"]
)

base["perc_clima_risco_ult_6m"] = safe_div(
    base["clima_risco_ult_6m"],
    base["total_acidentes_ult_6m"]
)

base["velocidade_media_ult_6m"] = safe_div(
    base["velocidade_soma_ult_6m"],
    base["total_acidentes_ult_6m"]
)

base["num_units_medio_ult_6m"] = safe_div(
    base["num_units_soma_ult_6m"],
    base["total_acidentes_ult_6m"]
)

# Variáveis temporais do mês que será previsto
base["mes_previsao"] = base["MES_REF"].dt.month
base["ano_previsao"] = base["MES_REF"].dt.year

# ============================================================
# 4. Criar alvo futuro: próximos 3 meses
# ============================================================
# Para cada beat, a linha do mês t recebe o score de t, t+1 e t+2.
# Isso representa a prioridade observada no período futuro.

base["score_futuro_3m"] = (
    base.groupby("BEAT_OF_OCCURRENCE")["score_prioridade_mes"]
    .transform(lambda s: s + s.shift(-1) + s.shift(-2))
)

# Também podemos guardar os acidentes futuros para análise
base["acidentes_futuros_3m"] = (
    base.groupby("BEAT_OF_OCCURRENCE")["total_acidentes_mes"]
    .transform(lambda s: s + s.shift(-1) + s.shift(-2))
)

base["mortes_futuras_3m"] = (
    base.groupby("BEAT_OF_OCCURRENCE")["total_mortes_mes"]
    .transform(lambda s: s + s.shift(-1) + s.shift(-2))
)

base["lesoes_graves_futuras_3m"] = (
    base.groupby("BEAT_OF_OCCURRENCE")["total_lesoes_graves_mes"]
    .transform(lambda s: s + s.shift(-1) + s.shift(-2))
)

# ============================================================
# 5. Remover linhas sem histórico suficiente ou sem futuro
# ============================================================

base_modelo = base.dropna(subset=[
    "total_acidentes_ult_6m",
    "score_futuro_3m"
]).copy()

# ============================================================
# 6. Criar classe de prioridade futura
# ============================================================
# Importante:
# Os cortes devem ser calculados usando apenas o período de treino,
# para evitar vazamento do conjunto de teste.

data_corte_treino = pd.Timestamp("2022-12-01")

treino_mask = base_modelo["MES_REF"] <= data_corte_treino

# Caso sua base não tenha dados suficientes até 2022, usar corte automático
if treino_mask.sum() == 0:
    meses_ordenados = sorted(base_modelo["MES_REF"].unique())
    data_corte_treino = meses_ordenados[int(len(meses_ordenados) * 0.7)]
    treino_mask = base_modelo["MES_REF"] <= data_corte_treino

q50 = base_modelo.loc[treino_mask, "score_futuro_3m"].quantile(0.50)
q80 = base_modelo.loc[treino_mask, "score_futuro_3m"].quantile(0.80)

def classificar_prioridade(score):
    if score <= q50:
        return "BAIXA"
    elif score <= q80:
        return "MEDIA"
    else:
        return "ALTA"

base_modelo["prioridade_futura"] = base_modelo["score_futuro_3m"].apply(classificar_prioridade)

# ============================================================
# 7. Selecionar features finais do classificador
# ============================================================

features_modelo = [
    "total_acidentes_ult_6m",
    "media_acidentes_mensal_ult_6m",

    "perc_acidentes_noite_ult_6m",
    "perc_acidentes_fim_semana_ult_6m",

    "perc_iluminacao_risco_ult_6m",
    "perc_pista_risco_ult_6m",
    "perc_defeito_via_ult_6m",
    "perc_dispositivo_problema_ult_6m",
    "perc_clima_risco_ult_6m",

    "velocidade_media_ult_6m",
    "num_units_medio_ult_6m",

    "mes_previsao"
]

colunas_finais = [
    "BEAT_OF_OCCURRENCE",
    "MES_REF",
    "lat_media_beat",
    "lon_media_beat",
] + features_modelo + [
    "score_futuro_3m",
    "prioridade_futura",
    "acidentes_futuros_3m",
    "mortes_futuras_3m",
    "lesoes_graves_futuras_3m"
]

base_classificacao = base_modelo[colunas_finais].copy()

# Salvar base final
base_classificacao.to_csv("base_classificacao_beat_mes.csv", index=False)

print("Base final do classificador criada:")
print(base_classificacao.shape)

print("\nDistribuição das classes:")
print(base_classificacao["prioridade_futura"].value_counts())

print("\nAmostra:")
print(base_classificacao.head())

Base final do classificador criada:
(26384, 21)

Distribuição das classes:
prioridade_futura
BAIXA    13292
MEDIA     7833
ALTA      5259
Name: count, dtype: int64

Amostra:
  BEAT_OF_OCCURRENCE    MES_REF  lat_media_beat  lon_media_beat  \
3               1011 2018-04-01         41.8643      -87.725041   
4               1011 2018-05-01         41.8643      -87.725041   
5               1011 2018-06-01         41.8643      -87.725041   
6               1011 2018-07-01         41.8643      -87.725041   
7               1011 2018-08-01         41.8643      -87.725041   

   total_acidentes_ult_6m  media_acidentes_mensal_ult_6m  \
3                    61.0                      10.166667   
4                    79.0                      13.166667   
5                   111.0                      18.500000   
6                   133.0                      22.166667   
7                   134.0                      22.333333   

   perc_acidentes_noite_ult_6m  perc_acidentes_fim_semana_ult_

In [17]:
base_classificacao.head()

,BEAT_OF_OCCURRENCE,MES_REF,lat_media_beat,lon_media_beat,total_acidentes_ult_6m,media_acidentes_mensal_ult_6m,perc_acidentes_noite_ult_6m,perc_acidentes_fim_semana_ult_6m,perc_iluminacao_risco_ult_6m,perc_pista_risco_ult_6m,...,perc_dispositivo_problema_ult_6m,perc_clima_risco_ult_6m,velocidade_media_ult_6m,num_units_medio_ult_6m,mes_previsao,score_futuro_3m,prioridade_futura,acidentes_futuros_3m,mortes_futuras_3m,lesoes_graves_futuras_3m
3,1011,2018-04-01,41.8643,-87.725041,61.0,10.166667,0.327869,0.213115,0.409836,0.327869,...,0.049180,0.278689,28.278689,1.983607,4,208.0,MEDIA,72.0,0.0,11.0
4,1011,2018-05-01,41.8643,-87.725041,79.0,13.166667,0.392405,0.227848,0.455696,0.278481,...,0.037975,0.253165,27.974684,1.962025,5,210.0,MEDIA,77.0,0.0,11.0
5,1011,2018-06-01,41.8643,-87.725041,111.0,18.500000,0.387387,0.234234,0.423423,0.234234,...,0.036036,0.207207,28.018018,1.981982,6,179.0,MEDIA,68.0,0.0,9.0
6,1011,2018-07-01,41.8643,-87.725041,133.0,22.166667,0.398496,0.225564,0.383459,0.233083,...,0.037594,0.203008,27.706767,1.984962,7,134.0,BAIXA,66.0,0.0,4.0
7,1011,2018-08-01,41.8643,-87.725041,134.0,22.333333,0.402985,0.246269,0.350746,0.171642,...,0.029851,0.171642,27.910448,1.977612,8,155.0,MEDIA,67.0,0.0,5.0
